In [2]:
import pandas as pd
import numpy as np
import json
import os
import re


# 读取并清洗数据
location_df = pd.read_csv('Location.csv')
location_df.rename(columns={'Geneid':'Gene'}, inplace=True)

ncbi_gene_id_df = pd.read_csv('NCBI_gene_id.csv', names=['NCBI_gene_id', 'Gene'])
ncbi_gene_id_df['Gene'] = ncbi_gene_id_df['Gene'].str.replace('mtm:','')
ncbi_gene_id_df['NCBI_gene_id'] = ncbi_gene_id_df['NCBI_gene_id'].str.replace('ncbi-geneid:','')

protein_id_df = pd.read_csv('Protein_id.csv', names=['Protein_id', 'Gene'])
protein_id_df['Gene'] = protein_id_df['Gene'].str.replace('mtm:','')
protein_id_df['Protein_id'] = protein_id_df['Protein_id'].str.replace('ncbi-proteinid:','')

uniprot_id_df = pd.read_csv('Uniprot_id.csv', names=['Uniport_id', 'Gene'])
uniprot_id_df['Gene'] = uniprot_id_df['Gene'].str.replace('mtm:','')
uniprot_id_df['Uniport_id'] = uniprot_id_df['Uniport_id'].str.replace('up:','')

kunm_df = pd.read_csv('Knum.csv', names=['Protein_id', 'Knum'])
kunm_df['Protein_id'] = kunm_df['Protein_id'].apply(lambda x: x[:-2])
kunm_df = kunm_df.groupby("Protein_id")["Knum"].apply(lambda x: ','.join(x)).reset_index()

description_df = pd.read_csv('Description.csv', names=['Protein_id', 'Description'])

pfam_df = pd.read_csv('PFAM.csv', names=['Protein_id', 'Pfam'])
pfam_df = pfam_df.groupby("Protein_id")["Pfam"].apply(lambda x: ','.join(x)).reset_index()

ec_number_df = pd.read_csv('EC_number.csv', names=['Protein_id', 'EC_number'])
ec_number_df = ec_number_df.groupby("Protein_id")["EC_number"].apply(lambda x: ','.join(x)).reset_index()

nucleic_acid_df = pd.read_csv('Nucleic_acid_sequence.csv')
nucleic_acid_df.rename(columns={'gene':'Gene','sequence':'Nucleic_acid_sequence'}, inplace=True)

amino_acid_sequence_df = pd.read_csv('Amino_acid_sequence.csv')
amino_acid_sequence_df.rename(columns={'Protein ID':'Protein_id','Sequence':'Amino_acid_sequence'}, inplace=True)


In [3]:
kegg_definition = pd.read_csv('KEGG_definition.csv')
kegg_definition.rename(columns={'Pathway':'KEGG_definition'}, inplace=True)

kegg_df = pd.read_csv('KEGG.csv', names=['KEGG', 'KEGG_definition'])

# 合并kegg_definition，kegg_df
kegg_merge_df = pd.merge(kegg_definition, kegg_df, on='KEGG_definition', how='inner')
kegg_merge_df = kegg_merge_df.fillna('')

kegg_merge_df = kegg_merge_df.groupby("Gene").agg({
    'KEGG_definition': lambda x: ';'.join(x),
    'KEGG': lambda x: ';'.join(x)
}).reset_index()

kegg_merge_df.to_csv('KEGG_merge.csv', index=False)
display(kegg_merge_df)


,Gene,KEGG_definition,KEGG
0,MYCTH_100519,Inositol phosphate metabolism;Metabolic pathwa...,mtm00562;mtm01100;mtm04070
1,MYCTH_100654,Autophagy - yeast;Protein processing in endopl...,mtm04138;mtm04141
2,MYCTH_101224,Fatty acid degradation;Tryptophan metabolism,mtm00071;mtm00380
3,MYCTH_101411,Aminoacyl-tRNA biosynthesis,mtm00970
4,MYCTH_101588,Inositol phosphate metabolism;Metabolic pathwa...,mtm00562;mtm01100;mtm04070;mtm04138
...,...,...,...
2091,MYCTH_t95,Aminoacyl-tRNA biosynthesis,mtm00970
2092,MYCTH_t96,Aminoacyl-tRNA biosynthesis,mtm00970
2093,MYCTH_t97,Aminoacyl-tRNA biosynthesis,mtm00970
2094,MYCTH_t98,Aminoacyl-tRNA biosynthesis,mtm00970


In [26]:
# Go.csv存在一对多情况
go_df = pd.read_csv('Go.csv')
go_df.rename(columns={'GOID':'GO', 'DEFINITION':'GO_definition', 'ONTOLOGY':'GO_type', 'TERM':'ONTOLOGY'}, inplace=True)
go_df = go_df.fillna('')

go_df = go_df.groupby('Gene').agg({
    'GO': lambda x: ';'.join(x),
    'GO_definition': lambda x: ';'.join(x),
    'ONTOLOGY': lambda x: ';'.join(x),
    'GO_type': lambda x: ';'.join(x)
}).reset_index()
go_df

,Gene,GO,GO_definition,ONTOLOGY,GO_type
0,MYCTH_100011,GO:0006281,The process of restoring DNA after damage. Gen...,DNA repair,BP
1,MYCTH_100068,GO:0005576;GO:0004553;GO:0005975;GO:0030248,The space external to the outermost structure ...,"extracellular region;hydrolase activity, hydro...",CC;MF;BP;MF
2,MYCTH_100089,GO:0005506;GO:0016702,Interacting selectively and non-covalently wit...,"iron ion binding;oxidoreductase activity, acti...",MF;MF
3,MYCTH_100290,GO:0000776;GO:0034508,A multisubunit complex that is located at the ...,kinetochore;centromere complex assembly,CC;BP
4,MYCTH_100401,GO:0016020;GO:0015205;GO:0022857;GO:0055085,A lipid bilayer along with all the proteins an...,membrane;nucleobase transmembrane transporter ...,CC;MF;MF;BP
...,...,...,...,...,...
5024,MYCTH_99686,GO:0005515,Interacting selectively and non-covalently wit...,protein binding,MF
5025,MYCTH_99786,GO:0004553;GO:0005975,Catalysis of the hydrolysis of any O-glycosyl ...,"hydrolase activity, hydrolyzing O-glycosyl com...",MF;BP
5026,MYCTH_99814,GO:0022857;GO:0055085,"Enables the transfer of a substance, usually a...",transmembrane transporter activity;transmembra...,MF;BP
5027,MYCTH_99838,GO:0009058;GO:0016747,The chemical reactions and pathways resulting ...,"biosynthetic process;transferase activity, tra...",BP;MF


In [36]:
df = pd.merge(location_df, ncbi_gene_id_df, on='Gene', how='outer')
df = pd.merge(df, protein_id_df, on='Gene', how='outer')
df = pd.merge(df, uniprot_id_df, on='Gene', how='outer')
df = pd.merge(df, kunm_df, on='Protein_id', how='outer')
df = pd.merge(df, description_df, on='Protein_id', how='outer')
df = pd.merge(df, pfam_df, on='Protein_id', how='outer')
df = pd.merge(df, ec_number_df, on='Protein_id', how='outer')
df = pd.merge(df, nucleic_acid_df, on='Gene', how='outer')
df = pd.merge(df, amino_acid_sequence_df, on='Protein_id', how='outer')
df = pd.merge(df, go_df, on='Gene', how='outer')
df = pd.merge(df, kegg_merge_df, on='Gene', how='outer')
df = df.fillna('N/A')

# 将df的Gene这一列按照升序排列，扩展到其他列
df = df.sort_values(by='Gene')
df = df.reset_index(drop=True)
display(df)

df.to_csv('gene_information.csv', index=False)

,Gene,Chr,Start,End,Strand,Length,NCBI_gene_id,Protein_id,Uniport_id,Knum,...,Pfam,EC_number,Nucleic_acid_sequence,Amino_acid_sequence,GO,GO_definition,ONTOLOGY,GO_type,KEGG_definition,KEGG
0,MYCTH_100011,chromosome: 1,6881744.0,6884436.0,-,2693.0,11505753,XP_003660026,G2Q701,N/A,...,2OG-FeII_Oxy_2,N/A,ATGAGTAACATCACCAACAAAGTGATTTCCCAGACGCTGTCCCCCG...,MSNITNKVISQTLSPVDVPPKTFTSSDTTKSRLAAQRLRIATKRAA...,GO:0006281,The process of restoring DNA after damage. Gen...,DNA repair,BP,N/A,N/A
1,MYCTH_100068,chromosome: 2,5265193.0,5266157.0,-,965.0,11512455,XP_003662402,G2Q913,K01181,...,"CBM_1,Glyco_hydro_11",3.2.1.8,ATGGTTGCTCTCTCTTCTCTCCTCGTCGCTGCCTCTGCGGCGGCCG...,MVALSSLLVAASAAAVAVAAPSEALQKRQTLTSSQTGFHDGFYYSF...,GO:0005576;GO:0004553;GO:0005975;GO:0030248,The space external to the outermost structure ...,"extracellular region;hydrolase activity, hydro...",CC;MF;BP;MF,N/A,N/A
2,MYCTH_100089,chromosome: 2,5175196.0,5176687.0,+,1492.0,11512426,XP_003662373,G2Q8Y3,N/A,...,Dioxygenase_C,N/A,ATGACGCTCAAGCAGTTGCTCATGAGCGCGCTGCTCGCAGCACCCG...,MTLKQLLMSALLAAPALAHPAHKEKVYAHAALPLERKSLAHCAAKF...,GO:0005506;GO:0016702,Interacting selectively and non-covalently wit...,"iron ion binding;oxidoreductase activity, acti...",MF;MF,N/A,N/A
3,MYCTH_100127,chromosome: 2,5035300.0,5035892.0,+,593.0,11512377,XP_003662324,G2Q8N0,K09780,...,YCII,N/A,ATGGCGACAGAAGCGCCCAAGAAGATGGAATGGCTCGTGGTCGTTC...,MATEAPKKMEWLVVVPDFPGAHEKRIAVRPQHFANLGPVKESGVFQ...,N/A,N/A,N/A,N/A,N/A,N/A
4,MYCTH_100290,chromosome: 2,4391122.0,4393687.0,+,2566.0,11512195,XP_003662142,G2Q7I4,K11501,...,CENP-I,N/A,ATGCCGTCCCCAGAGGATGATGACCTTGGCGGTCTCATCGGAGATC...,MPSPEDDDLGGLIGDLEAASKVPAKRRQTNIKSTVEKATALLYDRG...,GO:0000776;GO:0034508,A multisubunit complex that is located at the ...,kinetochore;centromere complex assembly,CC;BP,N/A,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9298,MYCTH_t95,chromosome: 3,1785962.0,1786056.0,-,95.0,11507534,N/A,N/A,N/A,...,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Aminoacyl-tRNA biosynthesis,mtm00970
9299,MYCTH_t96,chromosome: 3,1472358.0,1472429.0,-,72.0,11507441,N/A,N/A,N/A,...,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Aminoacyl-tRNA biosynthesis,mtm00970
9300,MYCTH_t97,chromosome: 3,942762.0,942861.0,-,100.0,11512736,N/A,N/A,N/A,...,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Aminoacyl-tRNA biosynthesis,mtm00970
9301,MYCTH_t98,chromosome: 3,86102.0,86183.0,-,82.0,11512532,N/A,N/A,N/A,...,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,Aminoacyl-tRNA biosynthesis,mtm00970


In [37]:
df_change = pd.read_csv('gene_information.csv')

df_dict = df_change.fillna('null').to_dict(orient='records')

In [39]:
output_dict = {}

for item in df_dict:
    Gene = 'KEY:' + item['Gene']
    output_dict[Gene] = {
        'Gene': item['Gene'],
        'NCBI_gene_id': int(item['NCBI_gene_id']) if item['NCBI_gene_id'] != 'null' else None,
        'Protein_id': item['Protein_id'],
        'Uniport_id': item['Uniport_id'],
        'Knum': str(item['Knum']),
        'Description': item['Description'],
        'PFAM': item['Pfam'],
        'Chr': item['Chr'],
        'Start': int(item['Start']) if item['Start'] != 'null' else None,
        'End': int(item['End']) if item['End'] != 'null' else None,
        'Strand': item['Strand'],
        'Length': int(item['Length']) if item['Length'] != 'null' else None,
        'Nucleic_acid_sequence': item['Nucleic_acid_sequence'],
        'Amino_acid_sequence': item['Amino_acid_sequence'],
        'GO_information': {
        'GO': item['GO'],
        'GO_definition': item['GO_definition'],
        'GO_type': item['GO_type'],
        },
        'KEGG': str(item['KEGG']),
        'KEGG_definition': str(item['KEGG_definition']),
        'EC_number': str(item['EC_number']),
    }

with open('gene_information.json', 'w') as f:
    json.dump(output_dict, f, indent=4)